# MoE 라우팅과 부하균형 실습 - Top-2 Router and Load Balance

- Tutorial ID: `mod-moe-routing-lab`
- Tutorial: MoE 라우팅과 부하균형 실습
- Section ID: `moe-1`
- Section: Top-2 Router and Load Balance


In [ ]:
# ============================================================
# 코드 읽는 법 — Top-2 Router and Load Balance
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) router가 토큰을 expert에 배정하는 과정과 load balancing 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(3)
T, d, E, k = 10, 6, 4, 2
X = np.random.randn(T, d)
Wg = np.random.randn(d, E) / np.sqrt(d)
logits = X @ Wg
probs = np.exp(logits - logits.max(axis=1, keepdims=True)); probs /= probs.sum(axis=1, keepdims=True)
top = np.argsort(-probs, axis=1)[:, :k]
counts = np.bincount(top.reshape(-1), minlength=E) / (T*k)
importance = probs.mean(axis=0)
aux_loss = E * np.sum(counts * importance)
print('top-k experts per token:
', top)
print('fraction routed:', np.round(counts, 3))
print('mean router prob:', np.round(importance, 3))
print('load-balance auxiliary loss:', round(aux_loss, 4))
print('expert collapse?', counts.max() > 0.6)